# Đánh giá marketing — so sánh model (Qwen2b / Qwen7b / GemmaE2B / GemmaE4B)

Tái sử dụng **cùng metric** như `marketing_finetune_evaluation.ipynb` (Faithfulness + Expansion + Marketing Vibe).

**Dữ liệu:** `dataset/high_quality_mock_cases_100.json` (cột `actual_output` = bài model sinh).

**Model sinh bài:** server local tương thích OpenAI — `LOCAL_LLM_BACKEND` (`lmstudio` / `ollama`). **LOCAL_MODEL_ID** + **LOCAL_OPENAI_BASE** tự gán từ `LMSTUDIO_*` hoặc `OLLAMA_*` tùy backend.

**Judge (DeepEval):** `OPENAI_API_KEY` **hoặc** `AZURE_OPENAI_ENDPOINT` + `AZURE_OPENAI_API_KEY`. Mặc định model judge: `gpt-5.4-mini` / `DEEPEVAL_JUDGE_MODEL`. Không dùng LM Studio làm judge.

**Mỗi lần chạy:** CSV tại `results/runs/…`, tổng hợp (append) tại `dataset/high_quality_mock_cases_100.eval_summary.csv`.

In [1]:
# %pip install -q deepeval openai pandas python-dotenv

In [2]:
import os
from pathlib import Path
from typing import Optional

from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
load_dotenv(_dotenv_path, override=True, encoding="utf-8")
ROOT = Path(_dotenv_path).resolve().parent if _dotenv_path else Path.cwd().resolve()

# ===== Hyperparameters — chỉnh tập trung tại đây =====

# Đường dẫn dữ liệu & output (tương đối ROOT)
DATASET_REL = "dataset/high_quality_mock_cases_100.json"
SUMMARY_REL = "results/high_quality_mock_cases_100.eval_summary.csv"
RUNS_REL = "results/runs"

# CSV ghi đè output theo case_id (None = chỉ dùng actual_output trong JSON)
OUTPUT_FROM_CSV: Optional[Path] = None

# Judge: OPENAI_API_KEY hoặc AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY
def _env(k: str) -> str:
    return (os.getenv(k) or "").strip()


JUDGE_MODEL = os.getenv("OPENAI_MODEL")
has_o, has_a = bool(_env("OPENAI_API_KEY")), bool(_env("AZURE_OPENAI_ENDPOINT") and _env("AZURE_OPENAI_API_KEY"))
if has_o and has_a:
    raise RuntimeError("Chỉ một: OPENAI_API_KEY hoặc cặp AZURE_OPENAI_* cho judge.")
if not has_o and not has_a:
    raise RuntimeError("Judge cần OPENAI_API_KEY hoặc AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY.")
if has_a:
    os.environ.setdefault("OPENAI_API_VERSION", "2024-08-01-preview")
    os.environ["USE_AZURE_OPENAI"] = "true"
    os.environ.pop("USE_OPENAI_MODEL", None)
    JUDGE_BACKEND = "azure"
else:
    os.environ.pop("USE_AZURE_OPENAI", None)
    JUDGE_BACKEND = "openai"
os.environ["OPENAI_MODEL"] = JUDGE_BACKEND

# Backend sinh bài (OpenAI-compatible): "lmstudio" | "ollama" — đặt trong .env: LOCAL_LLM_BACKEND=ollama
LOCAL_LLM_BACKEND = os.getenv("LOCAL_LLM_BACKEND", "lmstudio").strip().lower()
if LOCAL_LLM_BACKEND not in ("lmstudio", "ollama"):
    raise ValueError('LOCAL_LLM_BACKEND phải là "lmstudio" hoặc "ollama".')

# LM Studio / Ollama — LOCAL_MODEL_ID / LOCAL_OPENAI_BASE / LOCAL_API_KEY gán sau theo LOCAL_LLM_BACKEND
LMSTUDIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://127.0.0.1:1234/v1").rstrip("/")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434/v1").rstrip("/")

GENERATION_TEMPERATURE = 0.4

# Tiến trình đánh giá: in và đo thời gian mỗi N case (1–N, N+1–2N, …)
EVAL_PROGRESS_CHUNK = 10

DATASET_PATH = ROOT / DATASET_REL
SUMMARY_PATH = ROOT / SUMMARY_REL
RUNS_DIR = ROOT / RUNS_REL


def _to_openai_v1_base(url: str) -> str:
    b = url.rstrip("/")
    return b if b.endswith("/v1") else f"{b}/v1"


if LOCAL_LLM_BACKEND == "lmstudio":
    LOCAL_OPENAI_BASE = _to_openai_v1_base(LMSTUDIO_BASE_URL)
    LOCAL_MODEL_ID = (os.getenv("LMSTUDIO_MODEL_ID") or "local-model").strip() or "local-model"
    LOCAL_API_KEY = os.getenv("LMSTUDIO_API_KEY", "lm-studio")
elif LOCAL_LLM_BACKEND == "ollama":
    LOCAL_OPENAI_BASE = _to_openai_v1_base(OLLAMA_BASE_URL)
    LOCAL_MODEL_ID = (os.getenv("OLLAMA_MODEL_ID") or "llama3.2").strip() or "llama3.2"
    LOCAL_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")

print("JUDGE_MODEL:", JUDGE_MODEL, f"({JUDGE_BACKEND})")
print(
    "Sinh bài:",
    LOCAL_LLM_BACKEND,
    "| OpenAI-compatible:",
    LOCAL_OPENAI_BASE,
    "| LOCAL_MODEL_ID:",
    LOCAL_MODEL_ID,
)

JUDGE_MODEL: None (azure)
Sinh bài: ollama | OpenAI-compatible: http://192.168.92.26:11434/v1 | LOCAL_MODEL_ID: qwen3.5-2b


In [3]:
# Verify nhanh: Judge (OpenAI/Azure) + local LLM (LM Studio / Ollama) — phải chạy sau ô hyperparameters
from openai import AzureOpenAI, OpenAI

_hello_prompt = "Xin chào"

_judge = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/"),
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("OPENAI_API_VERSION", "2024-08-01-preview"),
)
_judge_model_id = JUDGE_MODEL


_r = _judge.chat.completions.create(
    model=_judge_model_id,
    messages=[{"role": "user", "content": _hello_prompt}],
)
print("Judge:", (_r.choices[0].message.content or "").strip())

NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

In [ ]:
_local = OpenAI(base_url=LOCAL_OPENAI_BASE, api_key=LOCAL_API_KEY)
_r2 = _local.chat.completions.create(
    model=LOCAL_MODEL_ID,
    messages=[{"role": "user", "content": _hello_prompt}],
)
print("Local:", (_r2.choices[0].message.content or "").strip())

In [ ]:
import json
import re
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from dotenv import find_dotenv, load_dotenv
from metrics import (
    JudgeLLM,
    FaithfulnessEvaluator,
    ExpansionQualityEvaluator,
    MarketingVibeEvaluator,
    run_batch,
)

load_dotenv(find_dotenv(usecwd=True), override=True, encoding="utf-8")
_o = (os.getenv("OPENAI_API_KEY") or "").strip()
_az = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip() and (os.getenv("AZURE_OPENAI_API_KEY") or "").strip()
if not (_o or _az):
    raise RuntimeError("Chạy ô hyperparameters trước hoặc đặt OPENAI_API_KEY hoặc AZURE_OPENAI_* trong .env.")
print("DeepEval metrics: sẵn sàng (judge =", os.environ.get("OPENAI_MODEL", "?"), ").")

judge_llm = JudgeLLM(_judge, _judge_model_id)
print(f"JudgeLLM: {judge_llm.get_model_name()}")

## Load dataset

Hyperparameter ở **ô đầu** (sau `%pip`). ~**3 lời gọi judge / case**. Tổng hợp: **CSV** `eval_summary.csv`.

In [ ]:
def load_cases(json_path: Path, override_csv: Optional[Path]) -> List[Dict[str, str]]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Dataset phải là mảng JSON.")
    cases = []
    for row in data:
        cases.append(
            {
                "case_id": str(row["case_id"]),
                "input_title": str(row["input_title"]),
                "seed_content": str(row["seed_content"]),
                "actual_output": str(row.get("actual_output", "")),
            }
        )
    if override_csv and override_csv.is_file():
        df_o = pd.read_csv(override_csv)
        if "case_id" not in df_o.columns or "actual_output" not in df_o.columns:
            raise ValueError("CSV override cần cột: case_id, actual_output")
        m = dict(zip(df_o["case_id"].astype(str), df_o["actual_output"].astype(str)))
        for c in cases:
            if c["case_id"] in m:
                c["actual_output"] = m[c["case_id"]]
    return cases


cases = load_cases(DATASET_PATH, OUTPUT_FROM_CSV)
print(f"Cases: {len(cases)} | dataset: {DATASET_PATH.name} | LOCAL_MODEL_ID: {LOCAL_MODEL_ID}")

### (Tùy chọn) Sinh `actual_output` qua LM Studio hoặc Ollama

Chạy **sau** ô load `cases`, **trước** ô `run_batch`. Client sinh bài: `LOCAL_OPENAI_BASE`, `LOCAL_MODEL_ID`, `LOCAL_API_KEY` (tự gán theo `LOCAL_LLM_BACKEND`) từ ô hyperparameters đầu tiên.

In [ ]:
# Uncomment để sinh bài qua local OpenAI-compatible (LM Studio hoặc Ollama — xem LOCAL_LLM_BACKEND)
# from openai import OpenAI
#
# _gen = OpenAI(base_url=LOCAL_OPENAI_BASE, api_key=LOCAL_API_KEY)
#
# def build_user_prompt(title: str, seed: str) -> str:
#     return f"Viết bài marketing Markdown từ tiêu đề và mồi:\nTiêu đề: {title}\nMồi: {seed}"
#
# for c in cases:
#     r = _gen.chat.completions.create(
#         model=LOCAL_MODEL_ID,
#         messages=[{"role": "user", "content": build_user_prompt(c["input_title"], c["seed_content"])}],
#         temperature=GENERATION_TEMPERATURE,
#     )
#     c["actual_output"] = (r.choices[0].message.content or "").strip()
print("(Khung sinh bài — đang tắt; bỏ comment khi cần.)")

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

safe_model = re.sub(r"[^A-Za-z0-9._-]+", "_", LOCAL_MODEL_ID).strip("_") or "model"
csv_path = RUNS_DIR / f"{safe_model}_{RUN_ID}.csv"

df_scores = run_batch(cases, judge_llm, progress_chunk=EVAL_PROGRESS_CHUNK)

base = pd.DataFrame([{**c} for c in cases])
# df_scores already contains case_id — drop it to avoid duplicate columns after concat
df_out = pd.concat(
    [base.reset_index(drop=True), df_scores.drop(columns=["case_id"]).reset_index(drop=True)],
    axis=1,
)
df_out.insert(0, "run_id", RUN_ID)
df_out.insert(1, "judge_backend", JUDGE_BACKEND)
df_out.insert(2, "judge_model", JUDGE_MODEL)
df_out.insert(3, "local_llm_backend", LOCAL_LLM_BACKEND)
df_out.insert(4, "local_model_id", LOCAL_MODEL_ID)
df_out.insert(5, "local_openai_base", LOCAL_OPENAI_BASE)

df_out.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {csv_path}")

In [ ]:
summary_cols = ["faithfulness_combined", "expansion_combined", "vibe_combined"]
means = df_scores[summary_cols].mean().to_dict()
overall = float(np.mean([means[c] for c in summary_cols]))

now = datetime.now(timezone.utc).isoformat()
summary_row = {
    "run_id": RUN_ID,
    "judge_backend": JUDGE_BACKEND,
    "judge_model": JUDGE_MODEL,
    "local_llm_backend": LOCAL_LLM_BACKEND,
    "local_model_id": LOCAL_MODEL_ID,
    "local_openai_base": LOCAL_OPENAI_BASE,
    "dataset_file": str(DATASET_PATH.relative_to(ROOT)),
    "n_cases": len(cases),
    "csv_file": str(csv_path.relative_to(ROOT)),
    "faithfulness_combined": means["faithfulness_combined"],
    "expansion_combined": means["expansion_combined"],
    "vibe_combined": means["vibe_combined"],
    "overall_mean": overall,
    "last_updated_utc": now,
}

new_df = pd.DataFrame([summary_row])
if SUMMARY_PATH.is_file():
    prev_df = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
    summary_out = pd.concat([prev_df, new_df], ignore_index=True)
else:
    summary_out = new_df
summary_out.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

print(f"Đã cập nhật tổng hợp: {SUMMARY_PATH}")
pd.DataFrame([summary_row]).T

In [ ]:
# Bảng so sánh nhanh theo model (từ file tổng hợp CSV)
pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig").sort_values(["local_model_id", "run_id"])